In [1]:
!pip install --quiet nltk scikit-learn pandas


In [4]:
import nltk

# Resource names have changed across NLTK versions -- download both old and new
# names and ignore failures, so this cell works regardless of your NLTK version.
resources = [
    "punkt", "punkt_tab",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker", "maxent_ne_chunker_tab",
    "words",
    "wordnet", "omw-1.4",
]

for resource in resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Skipped '{resource}': {e}")

print("Setup complete.")

Setup complete.


In [5]:
import nltk
from nltk import word_tokenize, pos_tag

sentence = "Apple is looking at buying a UK startup for $1 billion in London."
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)

for word, tag in tagged:
    print(f"{word:<12} {tag}")


Apple        NNP
is           VBZ
looking      VBG
at           IN
buying       VBG
a            DT
UK           NNP
startup      NN
for          IN
$            $
1            CD
billion      CD
in           IN
London       NNP
.            .


In [6]:
from nltk import ne_chunk

entree_tree = ne_chunk(tagged)
print(entree_tree)

for subtree in entree_tree:
    if hasattr(subtree, "label"):
      entity_text = " ".join([word for word, tag in subtree.leaves()])
      print(f"{entity_text:<20} -> {subtree.label()}")

(S
  (GPE Apple/NNP)
  is/VBZ
  looking/VBG
  at/IN
  buying/VBG
  a/DT
  UK/NNP
  startup/NN
  for/IN
  $/$
  1/CD
  billion/CD
  in/IN
  (GPE London/NNP)
  ./.)
Apple                -> GPE
London               -> GPE


In [7]:
from nltk import RegexpParser

chunk_grammar = "NP: {<DT>?<JJ>*<NN.*>+}"
chunk_parser = RegexpParser(chunk_grammar)

chunk_tree = chunk_parser.parse(tagged)
print(chunk_tree)


(S
  (NP Apple/NNP)
  is/VBZ
  looking/VBG
  at/IN
  buying/VBG
  (NP a/DT UK/NNP startup/NN)
  for/IN
  $/$
  1/CD
  billion/CD
  in/IN
  (NP London/NNP)
  ./.)


In [11]:
from nltk.corpus import wordnet as wn

synsets = wn.synsets("car")
print(f"Number of senses (synsets) for 'car': {len(synsets)}\n")

first = synsets[0]
print("Synset name:", first.name())
print("Definition :", first.definition())
print("Example    :", first.examples())
print("Synonyms   :", [lemma.name() for lemma in first.lemmas()])


Number of senses (synsets) for 'bank': 18

Synset name: bank.n.01
Definition : sloping land (especially the slope beside a body of water)
Example    : ['they pulled the canoe up on the bank', 'he sat on the bank of the river and watched the currents']
Synonyms   : ['bank']


In [12]:
from nltk.corpus import wordnet as wn

synsets = wn.synsets("bat")
print(f"Number of senses (synsets) for 'bat': {len(synsets)}\n")

first = synsets[0]
print("Synset name:", first.name())
print("Definition :", first.definition())
print("Example    :", first.examples())
print("Synonyms   :", [lemma.name() for lemma in first.lemmas()])


Number of senses (synsets) for 'bat': 10

Synset name: bat.n.01
Definition : nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate
Example    : []
Synonyms   : ['bat', 'chiropteran']


In [13]:
from nltk.corpus import wordnet as wn

synsets = wn.synsets("spring")
print(f"Number of senses (synsets) for 'spring': {len(synsets)}\n")

first = synsets[0]
print("Synset name:", first.name())
print("Definition :", first.definition())
print("Example    :", first.examples())
print("Synonyms   :", [lemma.name() for lemma in first.lemmas()])


Number of senses (synsets) for 'spring': 11

Synset name: spring.n.01
Definition : the season of growth
Example    : ['the emerging buds were a sure sign of spring', 'he will hold office until the spring of next year']
Synonyms   : ['spring', 'springtime']


In [19]:
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "The cat and dog are playful animals",
    "My dog loves to play in the park",
    "The new laptop has a fast processor",
    "This smartphone has an amazing camera",
    "The kitten chased the mouse around the house",
    "The software update improved battery life",
]
labels = [1, 1, 0, 0, 1, 0]

count_vectorizer = CountVectorizer(stop_words="english")
X = count_vectorizer.fit_transform(documents)

selector = SelectKBest(score_func=chi2, k=5)
selector.fit(X, labels)

selected_features = count_vectorizer.get_feature_names_out()[selector.get_support()]
print("Top 5 most informative words for this classification task:")
print(list(selected_features))

Top 5 most informative words for this classification task:
['dog', 'processor', 'smartphone', 'software', 'update']


In [23]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# -----------------------------
# Original documents
# -----------------------------

corpus = [
    "The stock market is doing well today.",
    "The stock market rose because of strong earnings.",
    "The chef prepared a delicious dinner.",
    "The restaurant served excellent food to its guests."
]


# -----------------------------
# TF-IDF Vectorization
# -----------------------------

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(corpus)


# -----------------------------
# Cosine Similarity
# -----------------------------

similarity_matrix = cosine_similarity(tfidf_matrix)


similarity_df = pd.DataFrame(
    similarity_matrix.round(2),
    columns=[f"doc {i+1}" for i in range(len(corpus))],
    index=[f"doc {i+1}" for i in range(len(corpus))]
)

print("Similarity Matrix:")
print(similarity_df)


# -----------------------------
# Try it on a new pair of documents
# -----------------------------

docs_to_compare = [
    "The stock market rallied today after strong earnings reports.",
    "Shares rose sharply following a positive quarterly earnings season.",
    "The chef prepared a delicious three-course dinner for the guests."
]


# Create TF-IDF matrix for new documents

tfidf2 = TfidfVectorizer()

matrix2 = tfidf2.fit_transform(docs_to_compare)


# Calculate cosine similarity

sim2 = cosine_similarity(matrix2)


# Display similarity matrix

similarity_df2 = pd.DataFrame(
    sim2.round(2),
    columns=[
        "doc A (market)",
        "doc B (earnings)",
        "doc C (dinner)"
    ],
    index=[
        "doc A (market)",
        "doc B (earnings)",
        "doc C (dinner)"
    ]
)

print("\nNew Documents Similarity Matrix:")
print(similarity_df2)

Similarity Matrix:
       doc 1  doc 2  doc 3  doc 4
doc 1   1.00   0.25   0.06   0.04
doc 2   0.25   1.00   0.05   0.04
doc 3   0.06   0.05   1.00   0.05
doc 4   0.04   0.04   0.05   1.00

New Documents Similarity Matrix:
                  doc A (market)  doc B (earnings)  doc C (dinner)
doc A (market)              1.00              0.07            0.13
doc B (earnings)            0.07              1.00            0.00
doc C (dinner)              0.13              0.00            1.00
